This file consists of code to load the video and looping through its frames

Imports:

In [1]:
import cv2
import os
import sys
from ultralytics import YOLO
from utils import VideoProcessor, BGRHandler
import numpy as np
import matplotlib.pyplot as plt

Loading the video

Notes:

I need to mask the frames before they get inferenced because i need the accuracy to go up

In [2]:

cap = cv2.VideoCapture(r'input-videos/08fd33_4.mp4')

# Get the default frame width and height
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, 20.0, (frame_width, frame_height))

model = YOLO("train64/weights/best.pt")
class_colors ={0: (0, 0, 255), 1: (0, 255, 0), 2: (255, 0, 0), 3:(0, 255, 255), 4:(255, 255, 0)}
processor = VideoProcessor(model, class_colors)
frame_counter = 0
while True:
    ret, frame = cap.read()
    if not ret:
        print("Finished reading video file. Exiting...")
        break
    if frame_counter == 0:
        converted_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots()
        ax.imshow(converted_frame) #when you do imshow, matplotlib creates a figure and an axes and event.xdata and event.ydata are the coordinates of the click in the axes coordinates
        bgr_handler = BGRHandler(frame)
        fig.canvas.mpl_connect('button_press_event', bgr_handler.click_event)
        plt.show()
        min_max_bgr_values = bgr_handler.min_max_bgr_values()
        frame_counter += 1
        
    mask = cv2.inRange(frame, min_max_bgr_values[0], min_max_bgr_values[1])
    masked_image = cv2.bitwise_and(frame, frame, mask=mask)
        
    frame = processor.process_frame(masked_image)
    out.write(frame)
cap.release()
out.release()

FileNotFoundError: [Errno 2] No such file or directory: 'train64\\weights\\best.pt'